# Hafta 2 · Hoca demosu: token ve sıcaklık

İki gösterim: Türkçe ile İngilizcenin token farkı (anahtar gerekmez) ve Görev B'deki sıcaklık deneyinin hazır örnek seti. Görev B öğrencide Google AI Studio'da yapılır; bu not defteri aynı istemi API'den üç sıcaklıkta çalıştırıp karşılaştırma için örnek üretir.

**Kullanım:** Colab'da açın ([bağlantı](https://colab.research.google.com/github/asimsinan/istem-muhendisligi-dersi/blob/main/hafta-02/demo.ipynb)), kurulum hücrelerini derse girmeden çalıştırın. Ders sırasında ilgili slaytın bölümüne gidip hücreyi çalıştırın. Toplam yaklaşık **10 istek** (ücretsiz günlük sınır 50; lab'larla aynı anahtar).

Model cevapları her çalıştırmada değişebilir. **Ders öncesi bir kez baştan sona çalıştırın**; beklenen fark çıkmazsa slayt notundaki “çıkmazsa” cümlesini kullanın.

## H2·3 · Bugünün sorusu: Türkçe kaç kat token?

Anahtar gerekmez. o200k (GPT-4o ve sonrası) ve cl100k (GPT-4) sözlükleriyle sayar. Slayttaki beklenen: o200k 26 / 17, cl100k 36 / 17.

In [ ]:
!pip -q install tiktoken
import tiktoken

TR = "Öğrencilerimizden gelen soruları cevaplayabilecek bir yapay zekâ asistanı geliştirmek istiyoruz."
EN = "We want to build an AI assistant that can answer the questions coming from our students."
for ad in ["o200k_base", "cl100k_base"]:
    enc = tiktoken.get_encoding(ad)
    tr, en = len(enc.encode(TR)), len(enc.encode(EN))
    print(f"{ad:12}  Türkçe {tr:3}   İngilizce {en:3}   oran {tr / en:.2f}")

enc = tiktoken.get_encoding("o200k_base")
print("\nTürkçe parçalar:", [enc.decode([t]) for t in enc.encode(TR)])

## Kurulum

Sol paneldeki 🔑 **Secrets**'ta `OPENROUTER_API_KEY` olsun ve *Notebook access* açık olsun. Bu iki hücreyi derse girmeden çalıştırın; ders sırasında yalnızca ilgili slaytın hücresini çalıştırırsınız.

H2'de öğrencilerin henüz OpenRouter anahtarı yok; bu bölüm yalnızca sizin anahtarınızla çalışır.

In [ ]:
!pip -q install openai

In [ ]:
import json, re, time
from openai import OpenAI, RateLimitError, APIStatusError, InternalServerError, APIConnectionError, APITimeoutError
from google.colab import userdata

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))
MODEL = "nvidia/nemotron-3.5-super-vl:free"   # yapılandırılmış çıktıyı destekler
MODEL_2 = "recursive/flint-v1.0:free"            # strict şemayı desteklemez
BEKLE = 3                                         # dakikada 20 istek sınırı


class GeciciHata(Exception):
    """Sağlayıcı 200 döndürdü ama cevap boş (ücretsiz modellerde ara sıra olur)."""


def iste(mesajlar, model=MODEL, sicaklik=0, **ek):
    """Tek istek; 429 ya da geçici sağlayıcı hatasında bekleyip yeniden dener. Cevap nesnesini döndürür."""
    govde = dict(ek.pop("extra_body", {}))
    govde.setdefault("reasoning", {"effort": "low"})
    for deneme in range(6):
        try:
            cevap = client.chat.completions.create(model=model, messages=mesajlar, temperature=sicaklik,
                                                   max_tokens=2000, extra_body=govde, **ek)
            if not cevap.choices:
                raise GeciciHata("boş cevap")
            time.sleep(BEKLE)
            return cevap
        except RateLimitError:
            bekle = 15 * (deneme + 1)
            print(f"429 · sınır ya da ücretsiz sağlayıcı yoğun; {bekle} sn bekleniyor ({deneme + 1}/6)")
        except (InternalServerError, APIConnectionError, APITimeoutError, GeciciHata) as h:
            bekle = 10 * (deneme + 1)
            print(f"Sağlayıcı geçici hata verdi ({type(h).__name__}); {bekle} sn sonra yeniden ({deneme + 1}/6)")
        time.sleep(bekle)
    raise RuntimeError("Üst üste hata: ücretsiz sağlayıcı yoğun ya da günlük sınır dolmuş olabilir. Birkaç dakika sonra hücreyi yeniden çalıştırın.")


def metin(c):
    return (c.choices[0].message.content or "").strip()


def json_bul(m):
    """Cevaptaki ilk {...} bloğunu okumayı dener; okuyamazsa None."""
    b = re.search(r"\{.*\}", m or "", re.S)
    try:
        return json.loads(b.group(0)) if b else None
    except json.JSONDecodeError:
        return None

print("Hazır.")

## H2·61 · Görev B: aynı istem, üç sıcaklık

AI Studio'daki deneyin API karşılığı: sıcaklık 0, 1 ve 2'de üçer kez (9 istek), sonra sıcaklık 2 + top-p 0,3 (1 istek). Model `flint-v1.0` (ücretsiz, hızlı; AI Studio'daki Gemma 4 31B OpenRouter'ın ücretsiz katmanında sürekli 429 verdiği için). Bu model düşünmeyi kapatmaya izin vermiyor; düşünme düşük tutulur.

Beklenen: 0'da üç cevap aynı ya da çok benzer; 1'de farklı ama makul; 2'de bazen tuhaf kelimeler. Top-p düşükken cevaplar toparlanır.

In [ ]:
ISTEM = "Isparta'da açılacak bir kafe için tek satırlık, akılda kalıcı bir slogan yaz. Yalnızca sloganı yaz."

for s in [0, 1, 2]:
    cevaplar = [metin(iste([{"role": "user", "content": ISTEM}], model=MODEL_2, sicaklik=s))
                for _ in range(3)]
    print(f"Sıcaklık {s}  ·  {len(set(cevaplar))} farklı cevap")
    for c in cevaplar:
        print("   ", c.replace("\n", " ")[:120])

c = iste([{"role": "user", "content": ISTEM}], model=MODEL_2, sicaklik=2, top_p=0.3)
print("Sıcaklık 2 + top-p 0,3:\n   ", metin(c)[:120])